In [1]:
## Install all required libraries before importing anything.
## thefuzz is used for fuzzy string matching in deduplication.
## python-Levenshtein speeds up thefuzz significantly.
!pip install lingua-language-detector -q
!pip install deep-translator -q
!pip install thefuzz -q
!pip install python-Levenshtein -q
!pip install faiss-cpu sentence-transformers vaderSentiment -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 11.6 MB/s eta 0:00:00


# Libraries and Imports

## Mount Drive and Raw Data

This connects the notebook to Google Drive
* Defines the two root directories
used throughout the project
* Creates the src package folder if it does not
exist yet
* Adds the package to the Python path so it can be imported.

**Run this first every time you open the notebook.**

In [2]:
from google.colab import drive
import os
import sys
import re
import unicodedata
import pandas as pd
import numpy as np
import faiss
from pathlib import Path
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer

drive.mount('/content/drive', force_remount=True)

# Define the source package directory and parent path
BASE_DIR = '/content/drive/Shareddrives/essentis_intern_drive'
DATA_DIR = os.path.join(BASE_DIR, 'data')
SRC_DIR  = os.path.join(DATA_DIR, 'src_reviews_cleaning')

# Add the data directory to sys.path so Python can find the package
if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)

print("SRC_DIR exists:      ", os.path.exists(SRC_DIR))
print("__init__.py exists:  ", os.path.exists(os.path.join(SRC_DIR, '__init__.py')))
print("DATA_DIR in sys.path:", DATA_DIR in sys.path)
print("\nFiles in SRC_DIR:")
for f in sorted(os.listdir(SRC_DIR)):
    if not f.startswith('__'):
        print(f"  {f}")

Mounted at /content/drive
SRC_DIR exists:       True
__init__.py exists:   True
DATA_DIR in sys.path: True

Files in SRC_DIR:
  README.md
  cleaners.py
  config.py
  deduplicator.py
  loaders.py
  normalizers.py
  pipeline.py
  requirements.txt


## Check Drive

In [3]:
# Check that Google Drive is mounted correctly and all expected paths exist.
# Run this cell at the start of any session before importing the pipeline.

import os

BASE_DIR   = '/content/drive/Shareddrives/essentis_intern_drive'
DATA_DIR   = os.path.join(BASE_DIR, 'data')
SRC_DIR    = os.path.join(DATA_DIR, 'src_reviews_cleaning')
RAW_DIR    = os.path.join(DATA_DIR, 'raw')
PROC_DIR   = os.path.join(DATA_DIR, 'processed')
CHARTS_DIR = os.path.join(BASE_DIR, 'charts')
NB_DIR     = os.path.join(BASE_DIR, 'colab_notebooks')

checks = {
    "Drive mounted":           os.path.exists('/content/drive/Shareddrives'),
    "Essentis drive found":    os.path.exists(BASE_DIR),
    "data/ exists":            os.path.exists(DATA_DIR),
    "data/raw/ exists":        os.path.exists(RAW_DIR),
    "data/processed/ exists":  os.path.exists(PROC_DIR),
    "src package exists":      os.path.exists(SRC_DIR),
    "__init__.py exists":      os.path.exists(os.path.join(SRC_DIR, '__init__.py')),
    "config.py exists":        os.path.exists(os.path.join(SRC_DIR, 'config.py')),
    "loaders.py exists":       os.path.exists(os.path.join(SRC_DIR, 'loaders.py')),
    "normalizers.py exists":   os.path.exists(os.path.join(SRC_DIR, 'normalizers.py')),
    "cleaners.py exists":      os.path.exists(os.path.join(SRC_DIR, 'cleaners.py')),
    "deduplicator.py exists":  os.path.exists(os.path.join(SRC_DIR, 'deduplicator.py')),
    "pipeline.py exists":      os.path.exists(os.path.join(SRC_DIR, 'pipeline.py')),
    "master_reviews_raw.csv":  os.path.exists(os.path.join(RAW_DIR, 'master_reviews_raw.csv')),
    "charts/ exists":          os.path.exists(CHARTS_DIR),
    "colab_notebooks/ exists": os.path.exists(NB_DIR),
}

all_passed = True
print("=== DRIVE MOUNT CHECK ===\n")
for label, result in checks.items():
    status = "✓" if result else "✗"
    if not result:
        all_passed = False
    print(f"  {status}  {label}")

print(f"\n{'All checks passed.' if all_passed else 'WARNING: some checks failed — fix before running the pipeline.'}")

if all_passed:
    print("\n=== FILES IN SRC PACKAGE ===")
    for f in sorted(os.listdir(SRC_DIR)):
        if not f.startswith('_'):
            print(f"  {f}")

    print("\n=== FILES IN RAW DATA ===")
    for f in sorted(os.listdir(RAW_DIR)):
        print(f"  {f}")

    print("\n=== FILES IN PROCESSED DATA ===")
    for f in sorted(os.listdir(PROC_DIR)):
        print(f"  {f}")

=== DRIVE MOUNT CHECK ===

  ✓  Drive mounted
  ✓  Essentis drive found
  ✓  data/ exists
  ✓  data/raw/ exists
  ✓  data/processed/ exists
  ✓  src package exists
  ✓  __init__.py exists
  ✓  config.py exists
  ✓  loaders.py exists
  ✓  normalizers.py exists
  ✓  cleaners.py exists
  ✓  deduplicator.py exists
  ✓  pipeline.py exists
  ✓  master_reviews_raw.csv
  ✓  charts/ exists
  ✓  colab_notebooks/ exists

All checks passed.

=== FILES IN SRC PACKAGE ===
  README.md
  cleaners.py
  config.py
  deduplicator.py
  loaders.py
  normalizers.py
  pipeline.py
  requirements.txt

=== FILES IN RAW DATA ===
  master_reviews_raw.csv

=== FILES IN PROCESSED DATA ===
  all_reviews_cleaned.csv
  dashboard_master.csv
  dashboard_master.gsheet
  dashboard_master1.gsheet


# Build Configurations
The single "source of truth" for the entire pipeline.
* All file paths, column names, and constants are defined here.
* Every other module imports from here - nothing is hard-coded anywhere else.

**If a column name or file path ever needs to change, change it here only.**

In [4]:
config_content = '''
from pathlib import Path
from typing import Final

# =============================================================================
# DIRECTORY PATHS
# Base directory for all data on the shared drive.
# All other paths are built relative to this.
# =============================================================================
BASE_DIR      = Path("/content/drive/Shareddrives/essentis_intern_drive")
RAW_DATA_DIR  = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# =============================================================================
# FILE PATHS
# Input and output file locations.
# PATH_MASTER_CSV is the single source of truth for raw data.
# =============================================================================
PATH_MASTER_CSV    = RAW_DATA_DIR / "master_reviews_raw.csv"
PATH_FINAL_CLEANED = PROCESSED_DIR / "all_reviews_cleaned.csv"

# =============================================================================
# COLUMN NAMES
# All column names used throughout the pipeline are defined here.
# Never hardcode column name strings anywhere else in the pipeline.
# =============================================================================
COL_BATCH_ID:               Final[str] = "batch_id"
COL_AUTHOR:                 Final[str] = "author"
COL_DATA_SOURCE:            Final[str] = "data_source"
COL_REVIEW_SOURCE:          Final[str] = "source"
COL_REVIEW_DATE:            Final[str] = "review_date"
COL_OVERALL_EXPERIENCE:     Final[str] = "overall_experience"
COL_REVIEW_RATING:          Final[str] = "review"
COL_INSTRUCTORS:            Final[str] = "instructors"
COL_CURRICULUM:             Final[str] = "curriculum"
COL_JOB_ASSISTANCE:         Final[str] = "job_assistance"
COL_REVIEW_TEXT:            Final[str] = "review_text"
COL_REVIEW_TEXT_TRANSLATED: Final[str] = "review_text_translated"
COL_COURSE:                 Final[str] = "course"
COL_COURSE_FORMAT:          Final[str] = "course_format"
COL_ROLE:                   Final[str] = "role"
COL_VERIFIED:               Final[str] = "verified"
COL_VERIFICATION_SOURCE:    Final[str] = "verification_source"
COL_LINK:                   Final[str] = "link"

# =============================================================================
# RATING COLUMNS
# The four numeric rating columns used to calculate overall_experience.
# =============================================================================
RATING_COLUMNS: Final[list] = [
    COL_REVIEW_RATING,
    COL_INSTRUCTORS,
    COL_CURRICULUM,
    COL_JOB_ASSISTANCE,
]

# =============================================================================
# VALIDATION CONSTANTS
# =============================================================================
MIN_RATING:  Final[int] = 1
MAX_RATING:  Final[int] = 5

# Date format used throughout the pipeline.
# YYYYMMDD format ensures consistent chronological sorting as a string.
DATE_FORMAT: Final[str] = "%Y%m%d"

# Fuzzy matching threshold for deduplication.
# Reviews with the same author and date whose text similarity exceeds
# this threshold are considered duplicates.
FUZZY_MATCH_THRESHOLD: Final[int] = 80
'''

config_path = os.path.join(SRC_DIR, 'config.py')
with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config_content.strip())

print("config.py written to:", config_path)

config.py written to: /content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/config.py


## Check .config

In [5]:
from importlib import reload
import src_reviews_cleaning.config as cfg
reload(cfg)

print("=== PATHS ===")
print(f"BASE_DIR:               {cfg.BASE_DIR}")
print(f"RAW_DATA_DIR:           {cfg.RAW_DATA_DIR}")
print(f"PROCESSED_DIR:          {cfg.PROCESSED_DIR}")
print(f"PATH_MASTER_CSV:        {cfg.PATH_MASTER_CSV}")
print(f"PATH_FINAL_CLEANED:     {cfg.PATH_FINAL_CLEANED}")

print("\n=== COLUMN NAMES ===")
all_cols = [
    cfg.COL_BATCH_ID, cfg.COL_AUTHOR, cfg.COL_DATA_SOURCE,
    cfg.COL_REVIEW_SOURCE, cfg.COL_REVIEW_DATE, cfg.COL_OVERALL_EXPERIENCE,
    cfg.COL_REVIEW_RATING, cfg.COL_INSTRUCTORS, cfg.COL_CURRICULUM,
    cfg.COL_JOB_ASSISTANCE, cfg.COL_REVIEW_TEXT, cfg.COL_REVIEW_TEXT_TRANSLATED,
    cfg.COL_COURSE, cfg.COL_COURSE_FORMAT, cfg.COL_ROLE,
    cfg.COL_VERIFIED, cfg.COL_VERIFICATION_SOURCE, cfg.COL_LINK,
]
for col in all_cols:
    print(f"  {col}")

print("\n=== CONSTANTS ===")
print(f"RATING_COLUMNS:        {cfg.RATING_COLUMNS}")
print(f"MIN_RATING:            {cfg.MIN_RATING}")
print(f"MAX_RATING:            {cfg.MAX_RATING}")
print(f"DATE_FORMAT:           {cfg.DATE_FORMAT}")
print(f"FUZZY_MATCH_THRESHOLD: {cfg.FUZZY_MATCH_THRESHOLD}")

dupes = len(all_cols) != len(set(all_cols))
print(f"\nDuplicate column names: {'YES - fix immediately' if dupes else 'none'}")
print("config.py check passed." if not dupes else "config.py check FAILED.")

=== PATHS ===
BASE_DIR:               /content/drive/Shareddrives/essentis_intern_drive
RAW_DATA_DIR:           /content/drive/Shareddrives/essentis_intern_drive/data/raw
PROCESSED_DIR:          /content/drive/Shareddrives/essentis_intern_drive/data/processed
PATH_MASTER_CSV:        /content/drive/Shareddrives/essentis_intern_drive/data/raw/master_reviews_raw.csv
PATH_FINAL_CLEANED:     /content/drive/Shareddrives/essentis_intern_drive/data/processed/all_reviews_cleaned.csv

=== COLUMN NAMES ===
  batch_id
  author
  data_source
  source
  review_date
  overall_experience
  review
  instructors
  curriculum
  job_assistance
  review_text
  review_text_translated
  course
  course_format
  role
  verified
  verification_source
  link

=== CONSTANTS ===
RATING_COLUMNS:        ['review', 'instructors', 'curriculum', 'job_assistance']
MIN_RATING:            1
MAX_RATING:            5
DATE_FORMAT:           %Y%m%d
FUZZY_MATCH_THRESHOLD: 80

Duplicate column names: none
config.py check passe

# Build Data Loaders

This reads each of the three raw data files we were given, and tags each row with a
data_source label so we always know where a row came from.

No cleaning happens here - just reading and tagging.

Sources:
    - reviews.json       (256 rows, Google Maps scrape)
    - reviews_clean.csv  (541 rows, cleaned by previous intern)
    - reviews_new.csv    (49 rows, most recent batch)


In [6]:
loaders_content = '''
import pandas as pd
from src_reviews_cleaning.config import (
    PATH_MASTER_CSV,
    COL_DATA_SOURCE,
)


def load_master_csv() -> pd.DataFrame:
    """
    Load the master reviews CSV from the shared drive.

    This file contains all three original data sources concatenated into
    one wide CSV, identified by the existing data_source column.
    The three sources are:
        - google: scraped from Google Maps (256 rows)
        - clean:  previously cleaned reviews from a prior intern (541 rows)
        - new:    most recent batch of reviews (49 rows)

    The file uses a semicolon separator and may contain multiline fields
    so the python engine is used for robustness.

    Args:
        None

    Returns:
        pd.DataFrame: Full master CSV with all 846 rows and 33 columns.

    Raises:
        FileNotFoundError: If the master CSV does not exist at PATH_MASTER_CSV.
        ValueError: If the data_source column is missing from the file.
    """
    df = pd.read_csv(
        PATH_MASTER_CSV,
        sep=";",
        engine="python",
        quotechar=\'"\',
    )
    if COL_DATA_SOURCE not in df.columns:
        raise ValueError(
            f"Expected column \'{COL_DATA_SOURCE}\' not found in master CSV. "
            f"Columns present: {df.columns.tolist()}"
        )
    return df


def load_all_raw() -> dict[str, pd.DataFrame]:
    """
    Load the master CSV and split it into separate DataFrames by data source.

    Reads the master CSV once and partitions it into three subsets based on
    the data_source column. Each subset is reset to a clean zero-based index.

    This is the main entry point used by the pipeline. Downstream normalizers
    expect the same column structure as the original raw sources so splitting
    here allows each normalizer to handle only its own columns.

    Args:
        None

    Returns:
        dict[str, pd.DataFrame]: Dictionary with keys "google", "clean", "new".
            Each value is a DataFrame containing only rows from that source
            with the index reset to start from zero.

    Raises:
        FileNotFoundError: If the master CSV does not exist at PATH_MASTER_CSV.
        ValueError: If the data_source column is missing from the file.
    """
    master = load_master_csv()
    return {
        "google": master[master[COL_DATA_SOURCE] == "google"].reset_index(drop=True),
        "clean":  master[master[COL_DATA_SOURCE] == "clean"].reset_index(drop=True),
        "new":    master[master[COL_DATA_SOURCE] == "new"].reset_index(drop=True),
    }
'''

loaders_path = os.path.join(SRC_DIR, 'loaders.py')
with open(loaders_path, 'w', encoding='utf-8') as f:
    f.write(loaders_content.strip())

print("loaders.py written to:", loaders_path)

loaders.py written to: /content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/loaders.py


## Check .loaders

In [7]:
from importlib import reload
import src_reviews_cleaning.loaders as L
reload(L)

raw = L.load_all_raw()

EXPECTED = {"google": 256, "clean": 541, "new": 49}
all_passed = True

print("=== RAW DATA SOURCES ===")
for source, df in raw.items():
    exp  = EXPECTED[source]
    ok   = df.shape[0] == exp
    if not ok:
        all_passed = False
    print(f"\n{source.upper()}")
    print(f"  Rows:        {df.shape[0]}  {'✓' if ok else f'✗ expected {exp}'}")
    print(f"  Cols:        {df.shape[1]}")
    print(f"  data_source: {df['data_source'].unique()}")

print(f"\nloaders.py check {'passed' if all_passed else 'FAILED'}.")

=== RAW DATA SOURCES ===

GOOGLE
  Rows:        256  ✓
  Cols:        33
  data_source: ['google']

CLEAN
  Rows:        541  ✓
  Cols:        33
  data_source: ['clean']

NEW
  Rows:        49  ✓
  Cols:        33
  data_source: ['new']

loaders.py check passed.


# Build Normalizers

* Renames each source's raw columns to the standard names defined in config.py.

* Drops irrelevant columns
* Combines review title and body into review_text
* Concatenates all three sources into one DataFrame.

Note: overall_experience is not calculated here. It is added in cleaners.py
after all individual rating columns have been cleaned.

In [8]:
normalizers_content = '''
import re
import pandas as pd
from src_reviews_cleaning.config import (
    COL_AUTHOR, COL_REVIEW_DATE, COL_INSTRUCTORS,
    COL_CURRICULUM, COL_JOB_ASSISTANCE, COL_REVIEW_TEXT, COL_COURSE_FORMAT,
    COL_ROLE, COL_VERIFIED, COL_VERIFICATION_SOURCE, COL_LINK,
    COL_REVIEW_SOURCE, COL_BATCH_ID, COL_DATA_SOURCE, COL_REVIEW_RATING,
    COL_COURSE, COL_REVIEW_TEXT_TRANSLATED,
)

# =============================================================================
# SOURCE-SPECIFIC RAW COLUMNS TO KEEP
# Because all sources are concatenated in one master CSV every row has all
# 33 columns. We keep only the columns relevant to each source before renaming
# to prevent duplicate column names after the rename step.
# =============================================================================
GOOGLE_KEEP = [
    "reviewer_name", "rating", "review_date", "review_text",
    "link", "photo_count", "review_images", "data_source",
]

CLEAN_KEEP = [
    "author", "date", "overall_experience", "instructors", "curriculum",
    "job_assistance", "title", "review", "verified", "verification_source",
    "role", "course", "format", "source", "Batch ID", "data_source",
]

NEW_KEEP = [
    "author", "date", "overall_experience", "instructors", "curriculum",
    "job_assistance", "title", "review", "course", "format",
    "source", "Batch ID", "data_source",
]

# =============================================================================
# FINAL COLUMN ORDER
# The standard set of columns every normalized DataFrame must have.
# COL_OVERALL_EXPERIENCE is intentionally excluded here - it is calculated
# in cleaners.py after all individual ratings have been cleaned.
# =============================================================================
FINAL_COLUMNS = [
    COL_BATCH_ID,
    COL_AUTHOR,
    COL_DATA_SOURCE,
    COL_REVIEW_SOURCE,
    COL_REVIEW_DATE,
    COL_REVIEW_RATING,
    COL_INSTRUCTORS,
    COL_CURRICULUM,
    COL_JOB_ASSISTANCE,
    COL_REVIEW_TEXT,
    COL_REVIEW_TEXT_TRANSLATED,
    COL_COURSE_FORMAT,
    COL_COURSE,
    COL_ROLE,
    COL_VERIFIED,
    COL_VERIFICATION_SOURCE,
    COL_LINK,
]


def _strip_emojis(text: str) -> str:
    """
    Remove all emoji and non-standard unicode characters from a string.

    Emojis cause rendering issues in matplotlib tables and CSV exports.
    This function removes any character whose unicode category starts with
    "So" (other symbol), "Cs" (surrogate), "Co" (private use), or "Cn"
    (unassigned), as well as any character above the Basic Multilingual
    Plane (code point > 0xFFFF) which covers most modern emoji.

    Standard Latin, accented characters, punctuation, and CJK characters
    are preserved.

    Args:
        text (str): Raw text potentially containing emoji characters.

    Returns:
        str: Clean text with all emoji characters removed.
             Returns the original value unchanged if it is not a string.
    """
    import unicodedata
    if not isinstance(text, str):
        return text
    return "".join(
        c for c in text
        if unicodedata.category(c) not in ("So", "Cs", "Co", "Cn")
        and ord(c) < 0x10000
    )


def _combine_title_and_review(df: pd.DataFrame, title_col: str, review_col: str) -> pd.Series:
    """
    Combine a title column and a review body column into a single text field.

    The clean and new sources store the review title and body in separate columns.
    This function merges them into one field for consistency with the Google source
    which only has a single review_text column.

    If a title exists it is prepended to the body with ": " as a separator.
    If no title exists the body is returned as-is.
    If both are empty or NaN the result is NaN.

    Args:
        df (pd.DataFrame): DataFrame containing both the title and review columns.
        title_col (str): Name of the column containing the review title.
        review_col (str): Name of the column containing the review body.

    Returns:
        pd.Series: Combined review text one value per row.
                   Rows with no title or body are returned as NaN.
    """
    title  = df[title_col].fillna("").str.strip()
    review = df[review_col].fillna("").str.strip()
    combined = title.where(title == "", title + ": ") + review
    return combined.replace("", pd.NA)


def _add_missing_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add any standard columns that are missing from a DataFrame as NaN.

    Ensures every normalized DataFrame has the full set of standard columns
    before they are concatenated in normalize_all().

    Args:
        df (pd.DataFrame): Partially normalized DataFrame missing some columns.

    Returns:
        pd.DataFrame: DataFrame with all FINAL_COLUMNS present.
    """
    for col in FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def normalize_google(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize raw Google Maps reviews to the standard column schema.

    Keeps only Google-specific columns to prevent duplicate column names,
    strips emojis from all text fields, renames columns to the standard
    schema, and hardcodes the review source as "google".

    Key differences from the standard schema:
        - reviewer name is in reviewer_name not author
        - star rating is in rating not review
        - review date is a relative string like "3 months ago"
        - no batch_id, role, verified, course, or course_format column
        - review source must be hardcoded as "google"

    Args:
        df (pd.DataFrame): Raw Google reviews subset from load_all_raw().
                           Expected shape: (256, 33).

    Returns:
        pd.DataFrame: Normalized Google reviews with all FINAL_COLUMNS present.
    """
    # Keep only Google-specific columns to avoid duplicates after renaming
    df = df[[c for c in GOOGLE_KEEP if c in df.columns]].copy()

    # Strip emojis from all object columns before any further processing
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(_strip_emojis)

    # Rename raw Google column names to standard pipeline names
    df = df.rename(columns={
        "reviewer_name": COL_AUTHOR,
        "rating":        COL_REVIEW_RATING,
        "review_date":   COL_REVIEW_DATE,
        "review_text":   COL_REVIEW_TEXT,
        "link":          COL_LINK,
    })

    # Drop columns not needed in the final schema
    df = df.drop(columns=["photo_count", "review_images"], errors="ignore")

    # Hardcode the review source since Google data has no source column
    df[COL_REVIEW_SOURCE] = "google"

    df = _add_missing_columns(df)
    return df[FINAL_COLUMNS]


def normalize_clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize the previously cleaned CSV reviews to the standard column schema.

    Keeps only clean-source columns, strips emojis, combines title and review
    body into review_text, and renames columns to the standard schema.

    Key differences from the standard schema:
        - review title and body are in separate columns (title, review)
        - date column is named date not review_date
        - overall_experience is used as the star rating column
        - course format is in format not course_format
        - reviewer batch ID is in Batch ID not batch_id

    Args:
        df (pd.DataFrame): Raw clean reviews subset from load_all_raw().
                           Expected shape: (541, 33).

    Returns:
        pd.DataFrame: Normalized clean reviews with all FINAL_COLUMNS present.
    """
    df = df[[c for c in CLEAN_KEEP if c in df.columns]].copy()

    # Strip emojis from all object columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(_strip_emojis)

    # Combine title and review body into one review_text field
    df[COL_REVIEW_TEXT] = _combine_title_and_review(df, title_col="title", review_col="review")
    df = df.drop(columns=["review", "title"], errors="ignore")

    df = df.rename(columns={
        "date":                COL_REVIEW_DATE,
        "overall_experience":  COL_REVIEW_RATING,
        "format":              COL_COURSE_FORMAT,
        "source":              COL_REVIEW_SOURCE,
        "Batch ID":            COL_BATCH_ID,
        "verification_source": COL_VERIFICATION_SOURCE,
        "course":              COL_COURSE,
    })

    df = _add_missing_columns(df)
    return df[FINAL_COLUMNS]


def normalize_new(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize the newest batch of reviews to the standard column schema.

    Keeps only new-source columns, strips emojis, combines title and review
    body into review_text, and renames columns to the standard schema.

    Key differences from the standard schema:
        - review title and body are in separate columns (title, review)
        - date column is named date not review_date
        - overall_experience is used as the star rating column
        - course format is in format not course_format
        - reviewer batch ID is in Batch ID not batch_id

    Args:
        df (pd.DataFrame): Raw new reviews subset from load_all_raw().
                           Expected shape: (49, 33).

    Returns:
        pd.DataFrame: Normalized new reviews with all FINAL_COLUMNS present.
    """
    df = df[[c for c in NEW_KEEP if c in df.columns]].copy()

    # Strip emojis from all object columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(_strip_emojis)

    # Combine title and review body into one review_text field
    df[COL_REVIEW_TEXT] = _combine_title_and_review(df, title_col="title", review_col="review")
    df = df.drop(columns=["review", "title"], errors="ignore")

    df = df.rename(columns={
        "date":               COL_REVIEW_DATE,
        "overall_experience": COL_REVIEW_RATING,
        "format":             COL_COURSE_FORMAT,
        "source":             COL_REVIEW_SOURCE,
        "Batch ID":           COL_BATCH_ID,
        "course":             COL_COURSE,
    })

    df = _add_missing_columns(df)
    return df[FINAL_COLUMNS]


def normalize_all(raw: dict) -> pd.DataFrame:
    """
    Normalize all three raw DataFrames and concatenate them into one.

    Calls each source-specific normalizer in sequence then stacks the
    results into a single DataFrame using pd.concat. The index is reset
    to a clean zero-based integer index after concatenation.

    Args:
        raw (dict): Output of load_all_raw(). Expected keys: "google", "clean", "new".

    Returns:
        pd.DataFrame: Single combined DataFrame with all reviews normalized.
                      Shape should be approximately (846, 17) before cleaning.

    Raises:
        KeyError: If any of the expected keys are missing from the raw dict.
    """
    normalized = [
        normalize_google(raw["google"]),
        normalize_clean(raw["clean"]),
        normalize_new(raw["new"]),
    ]
    return pd.concat(normalized, ignore_index=True)
'''

normalizers_path = os.path.join(SRC_DIR, 'normalizers.py')
with open(normalizers_path, 'w', encoding='utf-8') as f:
    f.write(normalizers_content.strip())

print("normalizers.py written to:", normalizers_path)

normalizers.py written to: /content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/normalizers.py


## Check .normalizers

In [9]:
from importlib import reload
import src_reviews_cleaning.loaders as L
import src_reviews_cleaning.normalizers as N
reload(L); reload(N)

raw = L.load_all_raw()

print("=== INDIVIDUAL NORMALIZERS ===")
google_norm = N.normalize_google(raw["google"])
clean_norm  = N.normalize_clean(raw["clean"])
new_norm    = N.normalize_new(raw["new"])

EXPECTED_SHAPES = {"google": (256, 17), "clean": (541, 17), "new": (49, 17)}
all_passed = True

for name, df in [("google", google_norm), ("clean", clean_norm), ("new", new_norm)]:
    exp      = EXPECTED_SHAPES[name]
    shape_ok = df.shape == exp
    if not shape_ok:
        all_passed = False
    # Check for duplicate column names
    dupes = [c for c in df.columns if df.columns.tolist().count(c) > 1]
    print(f"\n{name.upper()}")
    print(f"  Shape:      {df.shape}  {'✓' if shape_ok else f'✗ expected {exp}'}")
    print(f"  Duplicates: {dupes if dupes else 'none'}")
    print(f"  Course:     {df['course'].value_counts(dropna=False).to_dict()}")
    # Check emojis are stripped
    emoji_count = df.select_dtypes(include='object').apply(
        lambda col: col.dropna().apply(
            lambda x: any(ord(c) > 0x9FFF for c in str(x))
        ).sum()
    ).sum()
    print(f"  Emoji chars remaining: {emoji_count}")

print("\n=== COMBINED OUTPUT ===")
combined = N.normalize_all(raw)
print(f"Shape:   {combined.shape}")
print(f"Columns: {combined.columns.tolist()}")

print("\n=== COURSE VALUES ===")
print(combined['course'].value_counts(dropna=False).to_string())

print("\n=== DATA SOURCE COUNTS ===")
print(combined['data_source'].value_counts().to_string())

print(f"\nnormalizers.py check {'passed' if all_passed and combined.shape == (846, 17) else 'FAILED'}.")

=== INDIVIDUAL NORMALIZERS ===

GOOGLE
  Shape:      (256, 17)  ✓
  Duplicates: none
  Course:     {<NA>: 256}
  Emoji chars remaining: 2

CLEAN
  Shape:      (541, 17)  ✓
  Duplicates: none
  Course:     {'Full-Stack Web & App': 308, 'Data Science': 174, 'Unknown': 24, 'Product Design': 18, 'Marketing Analytics': 16, 'Full-Stack PHP Web Development': 1}
  Emoji chars remaining: 1

NEW
  Shape:      (49, 17)  ✓
  Duplicates: none
  Course:     {'Unknown': 23, 'Data Science': 11, 'Full-Stack Web & App': 8, 'Product Design': 6, 'Full-Stack PHP Web Development': 1}
  Emoji chars remaining: 0

=== COMBINED OUTPUT ===
Shape:   (846, 17)
Columns: ['batch_id', 'author', 'data_source', 'source', 'review_date', 'review', 'instructors', 'curriculum', 'job_assistance', 'review_text', 'review_text_translated', 'course_format', 'course', 'role', 'verified', 'verification_source', 'link']

=== COURSE VALUES ===
course
Full-Stack Web & App              316
NaN                               256
Data S

/content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/normalizers.py:297: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(normalized, ignore_index=True)


# Build Cleaner Functions
Step 5: Write cleaners.py

Cleans the values within each column. Each function handles one specific
cleaning job. No renaming or reshaping happens here.

* Ratings: handles European comma decimals, clamps to 1-5, casts to float
* overall_experience: calculated as the row-wise average of all four ratings
* verified: normalized to pandas BooleanDtype to support NaN alongside booleans
* course_format: standardized to full-time / part-time / NaN

**Dates: converts relative Google dates using a hardcoded scrape date of February 1 2025. *Update GOOGLE_SCRAPE_DATE in cleaners.py if the Google data is ever re-scraped*.**

In [10]:
cleaners_content = '''
import re
import time
import unicodedata
import pandas as pd
from datetime import datetime, timedelta
from lingua import Language, LanguageDetectorBuilder
from deep_translator import GoogleTranslator
from src_reviews_cleaning.config import (
    RATING_COLUMNS, COL_REVIEW_DATE, COL_VERIFIED, COL_OVERALL_EXPERIENCE,
    COL_COURSE_FORMAT, COL_COURSE, COL_ROLE, COL_REVIEW_TEXT,
    COL_REVIEW_TEXT_TRANSLATED, DATE_FORMAT, MIN_RATING, MAX_RATING,
)

# =============================================================================
# GOOGLE SCRAPE DATE ANCHOR
# Google reviews use relative dates like "3 months ago" instead of absolute
# dates. All relative dates are resolved by subtracting from this anchor date.
# IMPORTANT: If Google data is re-scraped update this date to match the new
# scrape date otherwise all Google dates will be wrong.
# =============================================================================
GOOGLE_SCRAPE_DATE = datetime(2025, 2, 1)

# =============================================================================
# COURSE NAME STANDARDIZATION MAP
# Maps raw course name strings to a fixed set of standard names.
# Keys are lowercase stripped versions of the raw values.
# Add new entries here if new courses are introduced.
# =============================================================================
COURSE_MAP = {
    "full-stack web & app":           "Full-Stack Web & App",
    "full-stack php web development": "Full-Stack PHP Development",
    "data science":                   "Data Science",
    "product design":                 "Product Design",
    "marketing analytics":            "Marketing Analytics",
}

# =============================================================================
# ROLE CATEGORIZATION KEYWORDS
# Maps raw role strings to broad categories using keyword matching.
# Order matters - the first matching category wins.
# =============================================================================
ROLE_KEYWORDS = {
    "Graduate":     ["graduate", "alumni"],
    "Student":      ["student"],
    "Applicant":    ["applicant"],
    "Developer":    ["developer", "engineer", "full stack", "fullstack", "full-stack", "frontend", "backend"],
    "Data Science": ["data scien", "analytics", "ml", "ai"],
    "Designer":     ["design"],
    "Manager":      ["manager", "consultant"],
}


def _parse_relative_date(relative: str, anchor: datetime) -> str:
    """
    Convert a relative date string to an absolute date string.

    Translates Google relative date format (e.g. "3 months ago") into a
    real date by subtracting the stated duration from the scrape anchor date.

    Args:
        relative (str): Relative date string from Google reviews.
                        Expected format: "<number|a> <unit>s? ago"
        anchor (datetime): The scrape date to calculate backwards from.

    Returns:
        str: Absolute date string formatted as DATE_FORMAT (YYYYMMDD),
             or pd.NA if the format is not recognized.
    """
    relative = str(relative).strip().lower()
    match = re.match("(a|[0-9]+)\\s+(day|week|month|year)s?\\s+ago", relative)
    if not match:
        return pd.NA
    quantity = 1 if match.group(1) == "a" else int(match.group(1))
    unit = match.group(2)
    if unit == "day":
        delta = timedelta(days=quantity)
    elif unit == "week":
        delta = timedelta(weeks=quantity)
    elif unit == "month":
        delta = timedelta(days=quantity * 30)
    elif unit == "year":
        delta = timedelta(days=quantity * 365)
    return (anchor - delta).strftime(DATE_FORMAT)


def clean_empty_strings(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert empty or whitespace-only strings to NaN across all object columns.

    Empty strings are not the same as NaN in pandas and can cause issues
    with language detection, word counts, and null value analysis.
    Only object dtype columns are touched.

    Args:
        df (pd.DataFrame): Combined normalized DataFrame from normalize_all().

    Returns:
        pd.DataFrame: DataFrame with empty strings replaced by pd.NA.
    """
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].replace("", pd.NA)
        df[col] = df[col].apply(
            lambda x: pd.NA if isinstance(x, str) and x.strip() == "" else x
        )
    return df


def clean_review_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize all review dates to pandas datetime64.

    Handles two date formats:
    1. Relative strings from Google (e.g. "3 months ago") resolved using
       _parse_relative_date() with GOOGLE_SCRAPE_DATE as anchor.
    2. Absolute date strings from CSV sources parsed directly.

    Final dates are stored as datetime64 so downstream analysis can use
    pandas date methods. The DATE_FORMAT constant (YYYYMMDD) is used for
    parsing relative dates before converting to datetime.

    Args:
        df (pd.DataFrame): Combined normalized DataFrame.

    Returns:
        pd.DataFrame: DataFrame with review_date as datetime64[ns].
    """
    def _convert(val: any) -> any:
        """Convert a single date value to datetime."""
        if pd.isna(val):
            return pd.NA
        val = str(val).strip()
        if "ago" in val:
            relative = _parse_relative_date(val, GOOGLE_SCRAPE_DATE)
            return pd.to_datetime(relative, format=DATE_FORMAT) if pd.notna(relative) else pd.NA
        try:
            return pd.to_datetime(val)
        except Exception:
            return pd.NA

    df[COL_REVIEW_DATE] = pd.to_datetime(
        df[COL_REVIEW_DATE].apply(_convert), errors="coerce"
    )
    return df


def detect_and_translate(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect the language of each review and translate non-English reviews.

    Uses lingua for language detection and deep-translator (Google Translate
    backend) for translation. Only non-English reviews are translated.
    The original review_text column is never modified. Translated text is
    stored in review_text_translated. A 2 second delay between translation
    calls avoids rate limiting.

    Args:
        df (pd.DataFrame): Combined normalized DataFrame after date cleaning.

    Returns:
        pd.DataFrame: DataFrame with review_text_translated populated for
                      non-English reviews. English reviews have NaN.
    """
    detector   = LanguageDetectorBuilder.from_all_languages().build()
    translator = GoogleTranslator(source="auto", target="en")
    translated = []
    total      = len(df)

    for i, text in enumerate(df[COL_REVIEW_TEXT]):
        if pd.isna(text):
            translated.append(pd.NA)
            continue
        language = detector.detect_language_of(str(text))
        if language is None or language == Language.ENGLISH:
            translated.append(pd.NA)
            continue
        try:
            result = translator.translate(str(text))
            translated.append(result)
            preview = str(text)[:50]
            print(f"  [{i+1}/{total}] Translated from {language.name}: {preview}...")
            time.sleep(2)
        except Exception as e:
            print(f"  [{i+1}/{total}] Translation failed: {e}")
            translated.append(pd.NA)

    df[COL_REVIEW_TEXT_TRANSLATED] = translated
    non_english = df[COL_REVIEW_TEXT_TRANSLATED].notna().sum()
    print(f"Translation complete. {non_english} non-English reviews translated.")
    return df


def clean_ratings(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize all numeric rating columns to float clamped to 1-5.

    Handles European comma decimals (e.g. "4,7" -> "4.7"), out-of-range
    values which are set to NaN, and non-numeric strings set to NaN.

    Args:
        df (pd.DataFrame): Combined normalized DataFrame.

    Returns:
        pd.DataFrame: DataFrame with all four rating columns as float64.
    """
    for col in RATING_COLUMNS:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )
        df[col] = df[col].where(df[col].between(MIN_RATING, MAX_RATING))
    return df


def calculate_overall_experience(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate overall_experience as the row-wise mean of all four rating columns.

    NaN values are ignored so partial ratings still produce a valid average.
    Rows where ALL four ratings are NaN receive NaN.
    Must be called after clean_ratings().

    Args:
        df (pd.DataFrame): DataFrame with all four rating columns cleaned.

    Returns:
        pd.DataFrame: DataFrame with overall_experience added as float64
                      rounded to 2 decimal places.
    """
    df[COL_OVERALL_EXPERIENCE] = df[RATING_COLUMNS].mean(axis=1).round(2)
    return df


def clean_verified(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize the verified column to pandas BooleanDtype.

    Uses nullable BooleanDtype so NaN coexists with True/False.
    Handles native booleans and string "TRUE"/"FALSE".

    Args:
        df (pd.DataFrame): Combined normalized DataFrame.

    Returns:
        pd.DataFrame: DataFrame with verified as pandas BooleanDtype.
    """
    def _convert(val: any) -> any:
        if pd.isna(val):
            return pd.NA
        if isinstance(val, bool):
            return val
        val_str = str(val).strip().upper()
        if val_str == "TRUE":
            return True
        if val_str == "FALSE":
            return False
        return pd.NA

    df[COL_VERIFIED] = df[COL_VERIFIED].apply(_convert).astype("boolean")
    return df


def clean_course_format(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize course_format to "full-time", "part-time", or NaN.

    Only the two valid formats are kept. Everything else is NaN.

    Args:
        df (pd.DataFrame): Combined normalized DataFrame.

    Returns:
        pd.DataFrame: DataFrame with course_format standardized.
    """
    VALID_FORMATS = {"full-time", "part-time"}

    def _convert(val: any) -> any:
        if pd.isna(val):
            return pd.NA
        val_lower = str(val).strip().lower()
        return val_lower if val_lower in VALID_FORMATS else pd.NA

    df[COL_COURSE_FORMAT] = df[COL_COURSE_FORMAT].apply(_convert)
    return df


def clean_course(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize the course column to a fixed set of known course names.

    Maps raw variations to standardized names using COURSE_MAP.
    Values of "Unknown" or anything not in COURSE_MAP are set to NaN.

    Args:
        df (pd.DataFrame): Combined normalized DataFrame.

    Returns:
        pd.DataFrame: DataFrame with course column standardized.
    """
    def _standardize(val: any) -> any:
        if pd.isna(val):
            return pd.NA
        val_lower = str(val).strip().lower()
        if val_lower == "unknown":
            return pd.NA
        return COURSE_MAP.get(val_lower, pd.NA)

    df[COL_COURSE] = df[COL_COURSE].apply(_standardize)
    return df


def clean_role(df: pd.DataFrame) -> pd.DataFrame:
    """
    Categorize the role column into a fixed set of broad role categories.

    Uses keyword matching via ROLE_KEYWORDS. The first matching category wins.
    Unmatched roles are categorized as "Other".

    Args:
        df (pd.DataFrame): Combined normalized DataFrame.

    Returns:
        pd.DataFrame: DataFrame with role column standardized.
    """
    def _categorize(val: any) -> any:
        if pd.isna(val):
            return pd.NA
        val_lower = str(val).strip().lower()
        for category, keywords in ROLE_KEYWORDS.items():
            if any(kw in val_lower for kw in keywords):
                return category
        return "Other"

    df[COL_ROLE] = df[COL_ROLE].apply(_categorize)
    return df


def clean_all(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run all cleaning functions on the combined normalized DataFrame.

    This is the main entry point used by pipeline.py. Steps run in order:
        1. clean_empty_strings       - standardize empty strings to NaN
        2. clean_review_dates        - parse all dates to datetime
        3. detect_and_translate      - translate non-English reviews (~90 sec)
        4. clean_ratings             - standardize all four numeric ratings
        5. calculate_overall_experience - average ratings after cleaning
        6. clean_verified            - normalize boolean column
        7. clean_course_format       - standardize format categories
        8. clean_course              - standardize course names
        9. clean_role                - categorize reviewer roles

    Args:
        df (pd.DataFrame): Combined normalized DataFrame from normalize_all().

    Returns:
        pd.DataFrame: Fully cleaned DataFrame ready for deduplication.
    """
    df = clean_empty_strings(df)
    df = clean_review_dates(df)
    df = detect_and_translate(df)
    df = clean_ratings(df)
    df = calculate_overall_experience(df)
    df = clean_verified(df)
    df = clean_course_format(df)
    df = clean_course(df)
    df = clean_role(df)
    return df
'''

cleaners_path = os.path.join(SRC_DIR, 'cleaners.py')
with open(cleaners_path, 'w', encoding='utf-8') as f:
    f.write(cleaners_content.strip())

print("cleaners.py written to:", cleaners_path)

cleaners.py written to: /content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/cleaners.py


## Check .cleaners

In [11]:
!pip install lingua-language-detector deep-translator -q

from importlib import reload
import src_reviews_cleaning.cleaners as C
reload(C)

import pandas as pd
test = combined.copy()

# clean_empty_strings
test = C.clean_empty_strings(test)
print("=== clean_empty_strings ===")
print(f"  Empty strings remaining: {(test.select_dtypes(include='object') == '').sum().sum()}")
print(f"  review_text NaNs:        {test['review_text'].isna().sum()}")

# clean_review_dates
test = C.clean_review_dates(test)
print("\n=== clean_review_dates ===")
print(f"  dtype:  {test['review_date'].dtype}")
print(f"  NaNs:   {test['review_date'].isna().sum()}")
print(f"  Min:    {test['review_date'].min()}")
print(f"  Max:    {test['review_date'].max()}")
print(f"  Sample Google dates: {test[test['data_source']=='google']['review_date'].head(3).tolist()}")

# detect_and_translate - 3 row sample only to avoid 90 second wait
print("\n=== detect_and_translate (3 row sample) ===")
german_sample = combined[
    combined['review_text'].str.lower().apply(
        lambda x: sum(p in str(x) for p in
            ['ich ', 'das ', 'die ', 'der ', 'und ', 'ist ']) >= 3
        if pd.notna(x) else False
    )
].head(3).copy()
german_sample = C.clean_empty_strings(german_sample)
german_sample = C.clean_review_dates(german_sample)
german_result = C.detect_and_translate(german_sample)
print(f"  Translated: {german_result['review_text_translated'].notna().sum()} / 3")
print(f"  Sample:     {str(german_result['review_text_translated'].iloc[0])[:100]}")

# clean_ratings
test = C.clean_ratings(test)
print("\n=== clean_ratings ===")
for col in ['review', 'instructors', 'curriculum', 'job_assistance']:
    print(f"  {col}: min={test[col].min()}  max={test[col].max()}  nulls={test[col].isna().sum()}")

# calculate_overall_experience
test = C.calculate_overall_experience(test)
print("\n=== calculate_overall_experience ===")
print(f"  min={test['overall_experience'].min()}  max={test['overall_experience'].max()}  NaNs={test['overall_experience'].isna().sum()}")

# clean_verified
test = C.clean_verified(test)
print("\n=== clean_verified ===")
print(f"  dtype:  {test['verified'].dtype}")
print(f"  values: {test['verified'].value_counts(dropna=False).to_dict()}")

# clean_course_format
test = C.clean_course_format(test)
print("\n=== clean_course_format ===")
print(f"  values: {test['course_format'].value_counts(dropna=False).to_dict()}")

# clean_course
test = C.clean_course(test)
print("\n=== clean_course ===")
print(test['course'].value_counts(dropna=False).to_string())

# clean_role
test = C.clean_role(test)
print("\n=== clean_role ===")
print(test['role'].value_counts(dropna=False).to_string())

print(f"\n=== FINAL SHAPE: {test.shape} ===")
print("cleaners.py check passed.")

/content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/cleaners.py:70: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("(a|[0-9]+)\s+(day|week|month|year)s?\s+ago", relative)


=== clean_empty_strings ===
  Empty strings remaining: 0
  review_text NaNs:        21

=== clean_review_dates ===
  dtype:  datetime64[ns]
  NaNs:   4
  Min:    2019-02-03 00:00:00
  Max:    2025-12-17 00:00:00
  Sample Google dates: [Timestamp('2025-01-02 00:00:00'), Timestamp('2025-01-02 00:00:00'), Timestamp('2025-01-02 00:00:00')]

=== detect_and_translate (3 row sample) ===
  [1/3] Translated from GERMAN: Der KI Kompaktkurs bei WBS Coding School bietet ei...
  [2/3] Translated from GERMAN: Abhängig vom eigenen Skillset und der jeweiligen G...
  [3/3] Translated from GERMAN: Der AI Compact-Kurs war ausgezeichnet. Er hat mir ...
Translation complete. 3 non-English reviews translated.
  Translated: 3 / 3
  Sample:     The AI ​​compact course at WBS Coding School offers a great introduction to current AI topics. The c

=== clean_ratings ===
  review: min=1.0  max=5.0  nulls=0
  instructors: min=1.0  max=5.0  nulls=580
  curriculum: min=1.0  max=5.0  nulls=256
  job_assistance: min=1.

# Build De-Duplicators

Finds and removes duplicate reviews.

* A duplicate is defined as two or more rows sharing the same **author, review_date, and review_text**.
* When duplicates are found, the row
with the fewest NaN values is kept.

In [12]:
deduplicator_content = '''
import pandas as pd
from thefuzz import fuzz
from src_reviews_cleaning.config import (
    COL_AUTHOR,
    COL_REVIEW_DATE,
    COL_REVIEW_TEXT,
    FUZZY_MATCH_THRESHOLD,
)


def _count_nulls(df: pd.DataFrame) -> pd.Series:
    """
    Count the number of NaN values in each row of a DataFrame.

    Used to rank duplicate rows so the most complete one is kept.

    Args:
        df (pd.DataFrame): Any DataFrame to count nulls across.

    Returns:
        pd.Series: Integer null count per row same index as df.
    """
    return df.isnull().sum(axis=1)


def _normalise_text(text: any) -> str:
    """
    Normalise a review text string for fuzzy comparison.

    Converts to lowercase and strips leading/trailing whitespace.
    This ensures minor capitalisation differences do not prevent
    fuzzy matching from identifying near-duplicate reviews.

    Args:
        text (any): Raw review text value possibly NaN.

    Returns:
        str: Normalised lowercase string or empty string if NaN.
    """
    if pd.isna(text):
        return ""
    return str(text).strip().lower()


def get_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return all exact duplicate rows for inspection without removing them.

    Uses exact matching on author + review_date + review_text.
    For fuzzy duplicates use remove_duplicates() which handles both.

    Args:
        df (pd.DataFrame): Cleaned DataFrame from clean_all().

    Returns:
        pd.DataFrame: All rows that have at least one exact duplicate
                      sorted so duplicate pairs appear next to each other.
                      Returns empty DataFrame if no exact duplicates found.
    """
    subset = [COL_AUTHOR, COL_REVIEW_DATE, COL_REVIEW_TEXT]
    dupes  = df[df.duplicated(subset=subset, keep=False)]
    return dupes.sort_values(subset).reset_index(drop=True)


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove duplicate reviews using a two-stage approach.

    Stage 1 — Exact deduplication:
        Removes rows where author, review_date, and review_text are all
        identical. When duplicates are found the row with the fewest NaN
        values is kept.

    Stage 2 — Fuzzy deduplication:
        Within each group of reviews sharing the same author and review_date,
        compares review texts using fuzzy string matching (token sort ratio).
        If two reviews from the same author on the same date have text
        similarity above FUZZY_MATCH_THRESHOLD (80%) they are considered
        duplicates and the more complete row is kept.

        Token sort ratio is used rather than simple ratio because it is
        robust to word reordering and handles cases where one review is
        a slightly rewritten version of another.

    The temporary _null_count column used for sorting is removed before
    returning so the output schema matches the input schema exactly.

    Args:
        df (pd.DataFrame): Cleaned DataFrame from clean_all().

    Returns:
        pd.DataFrame: DataFrame with duplicates removed and index reset.
                      The row with the fewest NaN values is kept from
                      each duplicate group.

    Side effects:
        Prints the number of exact and fuzzy duplicates removed
        and the final row count.
    """
    # -------------------------------------------------------------------------
    # Stage 1: Exact deduplication
    # -------------------------------------------------------------------------
    df["_null_count"] = _count_nulls(df)
    df = df.sort_values("_null_count", ascending=True)
    before_exact = len(df)
    df = df.drop_duplicates(
        subset=[COL_AUTHOR, COL_REVIEW_DATE, COL_REVIEW_TEXT],
        keep="first"
    )
    exact_removed = before_exact - len(df)
    print(f"Exact duplicates removed: {exact_removed}")

    # -------------------------------------------------------------------------
    # Stage 2: Fuzzy deduplication
    # -------------------------------------------------------------------------
    # Group by author and review_date - only compare within same author/date
    # to keep the operation tractable and avoid false positives
    df = df.reset_index(drop=True)
    indices_to_drop = set()

    # Get groups that have more than one review for the same author + date
    group_cols = [COL_AUTHOR, COL_REVIEW_DATE]
    grouped    = df.groupby(group_cols, dropna=False)

    for _, group in grouped:
        if len(group) < 2:
            # No potential duplicates in this group
            continue

        group_indices = group.index.tolist()

        # Compare every pair within the group
        for i in range(len(group_indices)):
            for j in range(i + 1, len(group_indices)):
                idx_a = group_indices[i]
                idx_b = group_indices[j]

                # Skip if one of the indices is already marked for removal
                if idx_a in indices_to_drop or idx_b in indices_to_drop:
                    continue

                text_a = _normalise_text(df.at[idx_a, COL_REVIEW_TEXT])
                text_b = _normalise_text(df.at[idx_b, COL_REVIEW_TEXT])

                # Empty texts cannot be meaningfully compared
                if not text_a or not text_b:
                    continue

                # Token sort ratio handles word reordering between versions
                similarity = fuzz.token_sort_ratio(text_a, text_b)

                if similarity >= FUZZY_MATCH_THRESHOLD:
                    # Keep the row with fewer nulls - drop the other
                    nulls_a = df.at[idx_a, "_null_count"]
                    nulls_b = df.at[idx_b, "_null_count"]
                    drop_idx = idx_b if nulls_a <= nulls_b else idx_a
                    indices_to_drop.add(drop_idx)

    fuzzy_removed = len(indices_to_drop)
    df = df.drop(index=list(indices_to_drop))
    print(f"Fuzzy duplicates removed: {fuzzy_removed} (threshold: {FUZZY_MATCH_THRESHOLD}%)")

    # Remove the temporary null count column
    df = df.drop(columns=["_null_count"])
    print(f"Rows remaining:           {len(df)}")

    return df.reset_index(drop=True)
'''

deduplicator_path = os.path.join(SRC_DIR, 'deduplicator.py')
with open(deduplicator_path, 'w', encoding='utf-8') as f:
    f.write(deduplicator_content.strip())

print("deduplicator.py written to:", deduplicator_path)

deduplicator.py written to: /content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/deduplicator.py


## Check .deduplicators

In [13]:
from importlib import reload
import src_reviews_cleaning.deduplicator as D
import src_reviews_cleaning.cleaners as C
reload(D); reload(C)

# Run all cleaning steps except translation for a fast check
test = combined.copy()
test = C.clean_empty_strings(test)
test = C.clean_review_dates(test)
test = C.clean_ratings(test)
test = C.calculate_overall_experience(test)
test = C.clean_verified(test)
test = C.clean_course_format(test)
test = C.clean_course(test)
test = C.clean_role(test)

print("=== get_duplicates (exact) ===")
exact_dupes = D.get_duplicates(test)
print(f"  Exact duplicate rows found: {len(exact_dupes)}")
if len(exact_dupes) > 0:
    print(exact_dupes[['author', 'review_date', 'data_source']].head(6).to_string(index=False))

print("\n=== remove_duplicates (exact + fuzzy) ===")
deduped = D.remove_duplicates(test.copy())
print(f"\n  Shape before: {test.shape}")
print(f"  Shape after:  {deduped.shape}")
print(f"  Index reset:  {deduped.index.tolist()[:5]}")

print("\n=== VALIDATION ===")
remaining_exact = D.get_duplicates(deduped)
null_col_gone   = "_null_count" not in deduped.columns
print(f"  Exact duplicates remaining: {len(remaining_exact)}")
print(f"  _null_count column removed: {null_col_gone}")

# Test fuzzy matching directly with a synthetic example
print("\n=== FUZZY MATCHING TEST ===")
from thefuzz import fuzz
text_a = "the course was excellent and i learned a lot"
text_b = "the course was excelent and i lerned alot"
sim = fuzz.token_sort_ratio(text_a, text_b)
print(f"  Similarity between near-identical texts: {sim}%")
print(f"  Would be caught as duplicate (>= 80%): {'Yes' if sim >= 80 else 'No'}")

all_passed = len(remaining_exact) == 0 and null_col_gone and sim >= 80
print(f"\ndeduplicator.py check {'passed' if all_passed else 'FAILED'}.")

=== get_duplicates (exact) ===
  Exact duplicate rows found: 6
   author review_date data_source
Anonymous  2025-01-17         new
Anonymous  2025-01-17         new
Anonymous  2025-07-08         new
Anonymous  2025-07-08         new
   Pia K.  2024-01-17       clean
   Pia K.  2024-01-17       clean

=== remove_duplicates (exact + fuzzy) ===
Exact duplicates removed: 3
Fuzzy duplicates removed: 4 (threshold: 80%)
Rows remaining:           839

  Shape before: (846, 18)
  Shape after:  (839, 18)
  Index reset:  [0, 1, 2, 3, 4]

=== VALIDATION ===
  Exact duplicates remaining: 0
  _null_count column removed: True

=== FUZZY MATCHING TEST ===
  Similarity between near-identical texts: 89%
  Would be caught as duplicate (>= 80%): Yes

deduplicator.py check passed.


# Build The Pipeline

The single entry point for the entire cleaning process.

* Calls each module in order and returns the final cleaned DataFrame.

* Optionally saves the result to a dated CSV file in PROCESSED_DIR.

**This is the only file the notebook needs to import to run the pipeline.**


In [14]:
pipeline_content = '''
import pandas as pd
from pathlib import Path
from src_reviews_cleaning.loaders import load_all_raw
from src_reviews_cleaning.normalizers import normalize_all
from src_reviews_cleaning.cleaners import clean_all
from src_reviews_cleaning.deduplicator import remove_duplicates
from src_reviews_cleaning.config import PROCESSED_DIR

# =============================================================================
# FINAL COLUMN ORDER
# Defines the exact column order of the pipeline output.
# All 18 columns are listed explicitly so the output is always predictable.
# If a new column is added to the pipeline it must be added here too.
# =============================================================================
FINAL_COLUMN_ORDER = [
    "batch_id",
    "author",
    "data_source",
    "source",
    "review_date",
    "overall_experience",
    "review",
    "instructors",
    "curriculum",
    "job_assistance",
    "review_text",
    "review_text_translated",
    "course_format",
    "course",
    "role",
    "verified",
    "verification_source",
    "link",
]


def run(save_csv: bool = False) -> pd.DataFrame:
    """
    Run the full reviews cleaning pipeline from raw data to clean output.

    This is the single entry point for the entire pipeline. It orchestrates
    all four modules in sequence and returns a fully cleaned deduplicated
    DataFrame ready for analysis.

    Pipeline steps:
        1. Load       - reads master_reviews_raw.csv and splits by data_source
        2. Normalize  - renames columns to standard schema per source
                        and strips emojis from all text fields
        3. Clean      - standardizes dates, ratings, text, course, role, and
                        detects and translates non-English reviews (~90 seconds)
        4. Deduplicate - removes exact duplicates and fuzzy near-duplicates
                         (80% similarity threshold on same author + date)
        5. Reorder    - applies FINAL_COLUMN_ORDER for consistent output

    Expected output:
        - Shape: approximately (843, 18)
        - All columns present and in FINAL_COLUMN_ORDER
        - review_date as datetime64
        - verified as pandas BooleanDtype
        - overall_experience as float64 rounded to 2 decimal places
        - review_text_translated populated for ~41 non-English reviews
        - No emoji characters in any text column

    Args:
        save_csv (bool): If True saves the cleaned DataFrame to
                         PROCESSED_DIR / "all_reviews_cleaned.csv".
                         Default is False.

    Returns:
        pd.DataFrame: Fully cleaned and deduplicated reviews DataFrame
                      with columns in FINAL_COLUMN_ORDER.

    Raises:
        FileNotFoundError: If master_reviews_raw.csv does not exist.
        KeyError: If expected columns are missing after normalization.

    Example:
        from src_reviews_cleaning.pipeline import run
        df = run()
        df = run(save_csv=True)
    """
    # Load raw data from master CSV and split by source
    print("Step 1/4: Loading raw data...")
    raw = load_all_raw()
    for source, frame in raw.items():
        print(f"  {source}: {frame.shape[0]} rows")

    # Normalize each source to the standard column schema
    print("Step 2/4: Normalizing columns...")
    combined = normalize_all(raw)
    print(f"  Combined shape: {combined.shape}")

    # Clean all columns including language detection and translation
    print("Step 3/4: Cleaning data (includes language detection + translation)...")
    cleaned = clean_all(combined)

    # Remove duplicate reviews using exact and fuzzy matching
    print("Step 4/4: Removing duplicates (exact + fuzzy)...")
    final = remove_duplicates(cleaned)

    # Reorder columns to the standard final order
    final = final[FINAL_COLUMN_ORDER]
    print(f"Pipeline complete. Final shape: {final.shape}")

    if save_csv:
        Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)
        output_path = Path(PROCESSED_DIR) / "all_reviews_cleaned.csv"
        final.to_csv(output_path, index=False)
        print(f"Saved to: {output_path}")

    return final
'''

pipeline_path = os.path.join(SRC_DIR, 'pipeline.py')
with open(pipeline_path, 'w', encoding='utf-8') as f:
    f.write(pipeline_content.strip())

print("pipeline.py written to:", pipeline_path)

pipeline.py written to: /content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/pipeline.py


## Run The Pipeline
**This will take longer** depending on the amount of non-english reviews detected.

In [15]:
from importlib import reload
import src_reviews_cleaning.pipeline as P
reload(P)

df = P.run(save_csv=True)

Step 1/4: Loading raw data...
  google: 256 rows
  clean: 541 rows
  new: 49 rows
Step 2/4: Normalizing columns...


/content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/normalizers.py:297: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(normalized, ignore_index=True)


  Combined shape: (846, 17)
Step 3/4: Cleaning data (includes language detection + translation)...
  [17/846] Translated from FRENCH: Très bonne école de formation. Les cours sont bien...
  [206/846] Translated from GERMAN: Der KI Kompaktkurs bei WBS Coding School bietet ei...
  [207/846] Translated from GERMAN: Abhängig vom eigenen Skillset und der jeweiligen G...
  [208/846] Translated from GERMAN: Der AI Compact-Kurs war ausgezeichnet. Er hat mir ...
  [209/846] Translated from GERMAN: Von Mai bis Juli 2025 habe ich an der WBS eine Wei...
  [210/846] Translated from GERMAN: Das Bootcamp "AI for Business" war für mich ein ec...
  [211/846] Translated from GERMAN: Ich habe den Data Science-Kurs abgeschlossen und e...
  [212/846] Translated from GERMAN: Anfangs war ich unsicher, ob mein Vorwissen für da...
  [213/846] Translated from GERMAN: Ich bin mit wenig Programmier-Erfahrung ins Data S...
  [214/846] Translated from GERMAN: Ich habe in den letzten 8 Wochen am KI-Kurs der WB...
  

## Check .pipeline

In [16]:
import os
from pathlib import Path
import src_reviews_cleaning.pipeline as P
import src_reviews_cleaning.deduplicator as D

print("=== PIPELINE OUTPUT ===")
print(f"  Shape: {df.shape}")

print("\n=== COLUMN ORDER ===")
col_order_ok = df.columns.tolist() == P.FINAL_COLUMN_ORDER
print(f"  Correct: {col_order_ok}")
if not col_order_ok:
    print(f"  Expected: {P.FINAL_COLUMN_ORDER}")
    print(f"  Actual:   {df.columns.tolist()}")

print("\n=== ROW COUNTS ===")
print(f"  Total: {len(df)}")
print(df['data_source'].value_counts().to_string())

print("\n=== DATE FORMAT ===")
print(f"  dtype:       {df['review_date'].dtype}")
print(f"  Min date:    {df['review_date'].min().date()}")
print(f"  Max date:    {df['review_date'].max().date()}")
print(f"  NaNs:        {df['review_date'].isna().sum()}")
print(f"  Sample dates:{df['review_date'].dropna().head(3).tolist()}")

print("\n=== TRANSLATIONS ===")
print(f"  Translated:   {df['review_text_translated'].notna().sum()}")
print(f"  Untranslated: {df['review_text_translated'].isna().sum()}")

print("\n=== EMOJI CHECK ===")
import unicodedata
def has_emoji(text: str) -> bool:
    """Check if a string contains any emoji characters."""
    if not isinstance(text, str):
        return False
    return any(
        unicodedata.category(c) in ("So", "Cs", "Co", "Cn") or ord(c) >= 0x10000
        for c in text
    )
emoji_count = df['review_text'].dropna().apply(has_emoji).sum()
print(f"  Reviews with emojis in review_text: {emoji_count}")

print("\n=== COURSE VALUES ===")
print(df['course'].value_counts(dropna=False).to_string())

print("\n=== RATINGS ===")
for col in ['review', 'instructors', 'curriculum', 'job_assistance', 'overall_experience']:
    print(f"  {col}: min={df[col].min()}  max={df[col].max()}  nulls={df[col].isna().sum()}")

print("\n=== NULLS PER COLUMN ===")
print(df.isnull().sum().to_string())

print("\n=== DUPLICATES REMAINING ===")
remaining = D.get_duplicates(df)
print(f"  {len(remaining)} exact duplicates remaining")

print("\n=== CSV SAVED ===")
csv_path = str(Path(P.PROCESSED_DIR) / "all_reviews_cleaned.csv")
print(f"  File exists: {os.path.exists(csv_path)}")
if os.path.exists(csv_path):
    import pandas as pd
    saved = pd.read_csv(csv_path)
    print(f"  Saved shape: {saved.shape}")

print(f"\npipeline.py check {'passed' if col_order_ok and emoji_count == 0 else 'FAILED'}.")

=== PIPELINE OUTPUT ===
  Shape: (839, 18)

=== COLUMN ORDER ===
  Correct: True

=== ROW COUNTS ===
  Total: 839
data_source
clean     536
google    256
new        47

=== DATE FORMAT ===
  dtype:       datetime64[ns]
  Min date:    2019-02-03
  Max date:    2025-12-17
  NaNs:        4
  Sample dates:[Timestamp('2023-09-07 00:00:00'), Timestamp('2023-09-05 00:00:00'), Timestamp('2024-09-10 00:00:00')]

=== TRANSLATIONS ===
  Translated:   40
  Untranslated: 799

=== EMOJI CHECK ===
  Reviews with emojis in review_text: 0

=== COURSE VALUES ===
course
Full-Stack Web & App          314
<NA>                          301
Data Science                  182
Product Design                 24
Marketing Analytics            16
Full-Stack PHP Development      2

=== RATINGS ===
  review: min=1.0  max=5.0  nulls=0
  instructors: min=1.0  max=5.0  nulls=573
  curriculum: min=1.0  max=5.0  nulls=256
  job_assistance: min=1.0  max=5.0  nulls=273
  overall_experience: min=1.0  max=5.0  nulls=0

=== N

# Exports

Create CSV for Dashboard use-
This will create a column that will be used for the dashboard.

The column will be populated by the statistical analysis, sentiment analysis, and student experience notebooks.

In [17]:
# Build and export dashboard_master.csv from the cleaned pipeline output.
# This cell must be run before any other notebook adds its columns.
# Run this cell any time the pipeline is re-run with new data.
# All columns are cast to explicit types before saving so Looker Studio
# reads every field correctly without manual type changes in the data source editor.

DASHBOARD_PATH = '/content/drive/Shareddrives/essentis_intern_drive/data/processed/dashboard_master.csv'

# =============================================================================
# FOUNDATION
# Start from the pipeline output and create the text column used throughout.
# Translated text is used for non-English reviews so all analysis runs on
# English text only.
# =============================================================================
dashboard_df = df.copy()

dashboard_df['text_for_analysis'] = dashboard_df['review_text_translated'].where(
    dashboard_df['review_text_translated'].notna(),
    dashboard_df['review_text']
)
dashboard_df['text_lower'] = dashboard_df['text_for_analysis'].str.lower()

print(f"Base shape: {dashboard_df.shape}")

# =============================================================================
# VADER SENTIMENT SCORING
# Scores each review on a scale of -1 (most negative) to +1 (most positive).
# Uses text_for_analysis so non-English reviews are scored on their
# English translations rather than the original text.
# =============================================================================
print("\nRunning VADER sentiment scoring...")
analyzer = SentimentIntensityAnalyzer()
scores = dashboard_df['text_for_analysis'].fillna("").apply(
    lambda x: analyzer.polarity_scores(str(x))
)
dashboard_df['sentiment_compound'] = scores.apply(lambda x: x['compound'])
dashboard_df['sentiment_pos']      = scores.apply(lambda x: x['pos'])
dashboard_df['sentiment_neg']      = scores.apply(lambda x: x['neg'])
dashboard_df['sentiment_neu']      = scores.apply(lambda x: x['neu'])
dashboard_df['sentiment_label']    = dashboard_df['sentiment_compound'].apply(
    lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
)
print(f"  Sentiment breakdown: {dashboard_df['sentiment_label'].value_counts().to_dict()}")

# =============================================================================
# REVIEW METADATA
# Text length and date parts for Looker Studio time series and length charts.
# =============================================================================
dashboard_df['review_length_chars'] = dashboard_df['text_for_analysis'].str.len()
dashboard_df['review_length_words'] = dashboard_df['text_for_analysis'].str.split().str.len()
dashboard_df['review_year']         = dashboard_df['review_date'].dt.year
dashboard_df['review_month']        = dashboard_df['review_date'].dt.to_period('M').astype(str)
dashboard_df['verified_label']      = dashboard_df['verified'].map(
    {True: 'Verified', False: 'Unverified'}
)

# =============================================================================
# ASPECT KEYWORD FLAGS
# 0/1 integer per review indicating whether each topic is mentioned.
# Stored as integers so Looker Studio can SUM them for frequency counts.
# =============================================================================
print("\nCalculating aspect flags...")
ASPECTS = {
    "teaching_quality":   ["instructor", "teacher", "mentor", "coaching", "explain", "taught", "lecture"],
    "curriculum":         ["curriculum", "content", "material", "syllabus", "project", "module", "topic"],
    "career_support":     ["job", "career", "hired", "employment", "placement", "interview", "linkedin"],
    "community":          ["community", "network", "peers", "classmates", "cohort", "group", "teammate"],
    "course_pace":        ["pace", "fast", "slow", "intense", "overwhelming", "rushed", "manageable"],
    "value_for_money":    ["price", "cost", "expensive", "worth", "money", "investment", "afford"],
    "online_experience":  ["online", "remote", "platform", "zoom", "virtual", "connection", "technical"],
    "support_and_staff":  ["support", "help", "staff", "response", "communication", "feedback", "admin"],
    "hardware_and_setup": ["hardware", "computer", "laptop", "equipment", "device", "setup", "tool"],
    "outcomes":           ["outcome", "result", "success", "achieve", "goal", "progress", "improvement"],
}
for aspect, keywords in ASPECTS.items():
    dashboard_df[f'aspect_{aspect}'] = dashboard_df['text_lower'].apply(
        lambda text: int(any(kw in str(text) for kw in keywords))
    )

# =============================================================================
# ENROLLMENT MOTIVATION FLAGS
# 0/1 integer per review indicating which motivation category is mentioned.
# =============================================================================
print("Calculating motivation flags...")
MOTIVATION_KEYWORDS = {
    "career_change":      ["career change", "career switch", "new career", "transition", "pivot"],
    "job_outcomes":       ["get a job", "find a job", "hired", "employment", "placement"],
    "curriculum_quality": ["curriculum", "up to date", "modern", "relevant", "practical"],
    "flexibility":        ["flexible", "part time", "part-time", "remote", "work while"],
    "reputation":         ["reputation", "recommended", "heard about", "word of mouth"],
    "price_value":        ["affordable", "price", "cost", "worth the money", "investment"],
    "community":          ["community", "network", "peers", "cohort", "support network"],
    "speed":              ["fast", "intensive", "short time", "quickly", "few months"],
}
for category, keywords in MOTIVATION_KEYWORDS.items():
    dashboard_df[f'motivation_{category}'] = dashboard_df['text_lower'].apply(
        lambda text: int(any(kw in str(text) for kw in keywords))
    )

# =============================================================================
# STUDENT EXPERIENCE FLAGS
# 0/1 integer per review for hiring mentions, USP language, expectation
# outcomes, and workload perception.
# =============================================================================
print("Calculating experience flags...")
EXPERIENCE_FLAGS = {
    'mentions_hiring': [
        "got a job", "found a job", "landed a job", "got hired", "been hired",
        "got an offer", "job offer", "employed", "working as", "now work",
        "currently work", "full time job", "started working", "new job",
    ],
    'mentions_usp': [
        "other schools", "compared to", "unlike other", "better than other",
        "only place", "only school", "best decision", "no regrets",
        "changed my life", "life changing", "transformed", "best investment",
    ],
    'expectation_exceeded': [
        "exceeded", "surpassed", "better than expected", "more than i expected",
        "pleasantly surprised", "beyond what i expected", "blown away",
    ],
    'expectation_fell_short': [
        "disappointed", "not what i expected", "expected more", "fell short",
        "not as good", "overpromised", "misleading", "not worth",
        "regret", "waste of money", "waste of time",
    ],
    'workload_heavy': [
        "overwhelming", "too much", "too many", "exhausting", "burnout",
        "burn out", "overloaded", "too demanding", "drowning", "struggled",
    ],
    'workload_manageable': [
        "manageable", "well paced", "well-paced", "balanced", "reasonable",
        "structured", "clear structure", "well structured", "well-structured",
    ],
}
for col, keywords in EXPERIENCE_FLAGS.items():
    dashboard_df[col] = dashboard_df['text_lower'].apply(
        lambda text: int(any(kw in str(text) for kw in keywords))
    )

# =============================================================================
# SEMANTIC SEARCH TAGGING
# Tags every review with its closest matching query, similarity score,
# and whether it matched a positive or negative theme.
# Must run after text_for_analysis is created.
# =============================================================================
print("\nRunning semantic search tagging...")

SEMANTIC_QUERIES = {
    "positive": [
        "positive career outcomes helpful job support successful hiring experience",
        "excellent teaching quality great instructors supportive mentors",
        "strong curriculum practical projects relevant content",
        "supportive community great peers positive atmosphere",
        "good value for money worth the investment affordable",
    ],
    "negative": [
        "career support issues weak job assistance poor interview help",
        "poor teaching quality unhelpful instructors bad mentoring",
        "outdated curriculum irrelevant content poor course structure",
        "technical issues poor platform bad online experience",
        "overwhelming workload too fast paced too intense",
    ],
}

all_queries  = []
query_themes = []
query_labels = []
for theme, queries in SEMANTIC_QUERIES.items():
    for q in queries:
        all_queries.append(q)
        query_themes.append(theme)
        query_labels.append(q[:60])

semantic_df            = dashboard_df.copy()
semantic_df['text_clean'] = (
    semantic_df['text_for_analysis']
    .fillna("").astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
has_text = semantic_df['text_clean'].str.len() > 0
print(f"  Reviews with text: {has_text.sum()} | Without text: {(~has_text).sum()}")

model      = SentenceTransformer("all-MiniLM-L6-v2")
review_emb = model.encode(
    semantic_df.loc[has_text, 'text_clean'].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
)
review_emb = np.array(review_emb, dtype="float32")
query_emb  = np.array(
    model.encode(all_queries, normalize_embeddings=True, show_progress_bar=False),
    dtype="float32"
)
query_index = faiss.IndexFlatIP(query_emb.shape[1])
query_index.add(query_emb)
scores_sem, indices_sem = query_index.search(review_emb, 1)

semantic_df.loc[has_text,  'semantic_best_score'] = scores_sem[:, 0]
semantic_df.loc[has_text,  'semantic_best_query'] = [query_labels[i] for i in indices_sem[:, 0]]
semantic_df.loc[has_text,  'semantic_theme']      = [query_themes[i] for i in indices_sem[:, 0]]
semantic_df.loc[~has_text, 'semantic_best_score'] = np.nan
semantic_df.loc[~has_text, 'semantic_best_query'] = pd.NA
semantic_df.loc[~has_text, 'semantic_theme']      = pd.NA

dashboard_df['semantic_best_score'] = semantic_df['semantic_best_score'].values
dashboard_df['semantic_best_query'] = semantic_df['semantic_best_query'].values
dashboard_df['semantic_theme']      = semantic_df['semantic_theme'].values

print(f"  Theme distribution: {dashboard_df['semantic_theme'].value_counts().to_dict()}")

# =============================================================================
# DROP INTERNAL COLUMNS
# text_lower was only needed for keyword matching and should not appear
# in the final dashboard file.
# =============================================================================
dashboard_df = dashboard_df.drop(columns=['text_lower'], errors='ignore')

# =============================================================================
# TYPE CASTING
# All columns are cast to explicit types before saving.
# This ensures Looker Studio reads every field correctly without needing
# manual type changes in the data source editor.
# =============================================================================
print("\nCasting column types...")

# Date — store as YYYYMMDD string so Looker reads it as a date
if 'review_date' in dashboard_df.columns:
    dashboard_df['review_date'] = pd.to_datetime(
        dashboard_df['review_date'], errors='coerce'
    ).dt.strftime('%Y%m%d')

# Flag columns — fill NaN with 0 and cast to int
FLAG_COLUMNS = (
    [f'aspect_{k}'    for k in ASPECTS] +
    [f'motivation_{k}' for k in MOTIVATION_KEYWORDS] +
    list(EXPERIENCE_FLAGS.keys())
)
for col in FLAG_COLUMNS:
    if col in dashboard_df.columns:
        dashboard_df[col] = dashboard_df[col].fillna(0).astype(int)

# Integer columns
for col in ['review_year', 'review_length_chars', 'review_length_words', 'dominant_topic']:
    if col in dashboard_df.columns:
        dashboard_df[col] = pd.to_numeric(
            dashboard_df[col], errors='coerce'
        ).astype('Int64')

# Float columns — rounded to sensible decimal places
for col, dp in {
    'overall_experience': 2, 'review': 2, 'instructors': 2,
    'curriculum': 2, 'job_assistance': 2, 'topic_weight': 2,
    'sentiment_compound': 3, 'sentiment_pos': 3,
    'sentiment_neg': 3, 'sentiment_neu': 3,
    'semantic_best_score': 3, 'readability_score': 1,
}.items():
    if col in dashboard_df.columns:
        dashboard_df[col] = pd.to_numeric(
            dashboard_df[col], errors='coerce'
        ).round(dp)

# batch_id — float, NaN allowed
if 'batch_id' in dashboard_df.columns:
    dashboard_df['batch_id'] = pd.to_numeric(dashboard_df['batch_id'], errors='coerce')

# Boolean verified — convert to plain text for Looker
if 'verified' in dashboard_df.columns:
    dashboard_df['verified'] = dashboard_df['verified'].map(
        {True: 'True', False: 'False', pd.NA: ''}
    ).fillna('')

# String columns — replace NaN and "nan" strings with empty string
STRING_COLUMNS = [
    'author', 'data_source', 'source', 'review_text', 'review_text_translated',
    'text_for_analysis', 'course_format', 'course', 'role', 'verification_source',
    'link', 'review_month', 'verified_label', 'sentiment_label',
    'semantic_best_query', 'semantic_theme',
]
for col in STRING_COLUMNS:
    if col in dashboard_df.columns:
        dashboard_df[col] = dashboard_df[col].fillna('').astype(str).replace('nan', '')

print("Type casting complete.")

# =============================================================================
# SAVE
# =============================================================================
Path(DASHBOARD_PATH).parent.mkdir(parents=True, exist_ok=True)
dashboard_df.to_csv(DASHBOARD_PATH, index=False)

print(f"\nSaved: dashboard_master.csv")
print(f"Shape: {dashboard_df.shape}")
print(f"\nColumn dtypes:")
print(dashboard_df.dtypes.to_string())

Base shape: (839, 20)

Running VADER sentiment scoring...
  Sentiment breakdown: {'positive': 791, 'neutral': 34, 'negative': 14}

Calculating aspect flags...
Calculating motivation flags...
Calculating experience flags...

Running semantic search tagging...
  Reviews with text: 818 | Without text: 21


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

  Theme distribution: {'positive': 699, 'negative': 119}

Casting column types...
Type casting complete.

Saved: dashboard_master.csv
Shape: (839, 56)

Column dtypes:
batch_id                         float64
author                            object
data_source                       object
source                            object
review_date                       object
overall_experience               float64
review                           float64
instructors                      float64
curriculum                       float64
job_assistance                   float64
review_text                       object
review_text_translated            object
course_format                     object
course                            object
role                              object
verified                          object
verification_source               object
link                              object
text_for_analysis                 object
sentiment_compound               float64
sentiment_pos

# Write Requirements

* Lists all third-party libraries needed to run the pipeline.
* Standard library modules are not listed here as they come built into Python.

Install all dependencies with:
    !pip install -r /content/drive/Shareddrive/src_reviews_cleaning/requirements.txt

In [18]:
requirements_content = """
pandas>=2.0.0
numpy>=1.24.0
vaderSentiment>=3.3.2
lingua-language-detector>=2.0.0
deep-translator>=1.11.0
thefuzz>=0.20.0
python-Levenshtein>=0.21.0
openpyxl>=3.1.0
"""

requirements_path = os.path.join(SRC_DIR, 'requirements.txt')
with open(requirements_path, 'w', encoding='utf-8') as f:
    f.write(requirements_content.strip())

print("requirements.txt written to:", requirements_path)
print("\nContents:")
print(requirements_content.strip())

requirements.txt written to: /content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/requirements.txt

Contents:
pandas>=2.0.0
numpy>=1.24.0
vaderSentiment>=3.3.2
lingua-language-detector>=2.0.0
deep-translator>=1.11.0
thefuzz>=0.20.0
python-Levenshtein>=0.21.0
openpyxl>=3.1.0


# Write README
Project documentation for the next intern (team).
* Covers the file structure
* How to run the pipeline
* The raw data sources
* The final column schema
* What each module does
* Important notes
* How to add a new data source in the future.

In [19]:
requirements_content = """
pandas>=2.0.0
numpy>=1.24.0
vaderSentiment>=3.3.2
lingua-language-detector>=2.0.0
deep-translator>=1.11.0
thefuzz>=0.20.0
python-Levenshtein>=0.21.0
openpyxl>=3.1.0
faiss-cpu>=1.7.4
sentence-transformers>=2.2.0
"""

requirements_path = os.path.join(SRC_DIR, 'requirements.txt')
with open(requirements_path, 'w', encoding='utf-8') as f:
    f.write(requirements_content.strip())

print("requirements.txt written to:", requirements_path)
print("\nContents:")
print(requirements_content.strip())

requirements.txt written to: /content/drive/Shareddrives/essentis_intern_drive/data/src_reviews_cleaning/requirements.txt

Contents:
pandas>=2.0.0
numpy>=1.24.0
vaderSentiment>=3.3.2
lingua-language-detector>=2.0.0
deep-translator>=1.11.0
thefuzz>=0.20.0
python-Levenshtein>=0.21.0
openpyxl>=3.1.0
faiss-cpu>=1.7.4
sentence-transformers>=2.2.0


In [20]:
import os

req_path    = os.path.join(SRC_DIR, 'requirements.txt')
readme_path = os.path.join(SRC_DIR, 'README.md')

print("=== requirements.txt ===")
print(f"  Exists: {os.path.exists(req_path)}")
with open(req_path, 'r') as f:
    req_content = f.read()
print(req_content)

required_packages = [
    'pandas', 'numpy', 'vaderSentiment', 'lingua-language-detector',
    'deep-translator', 'thefuzz', 'python-Levenshtein', 'openpyxl',
    'faiss-cpu', 'sentence-transformers',
]
print("Package checks:")
for pkg in required_packages:
    print(f"  {'✓' if pkg in req_content else '✗'}  {pkg}")

print("\n=== README.md ===")
print(f"  Exists: {os.path.exists(readme_path)}")
with open(readme_path, 'r') as f:
    readme = f.read()

sections = [
    "Quick Start",
    "File Structure",
    "Final Column Schema",
    "Dashboard Master CSV",
    "Semantic Search",
    "Language Detection and Translation",
    "Emoji Handling",
    "Fuzzy deduplication",
    "FUZZY_MATCH_THRESHOLD",
    "YYYYMMDD",
    "Run Order",
    "How To Add A New Data Source",
    "How To Update",
    "Dependencies",
    "GOOGLE_SCRAPE_DATE",
    "faiss-cpu",
    "sentence-transformers",
    "semantic_best_score",
    "semantic_theme",
]
print("\nSection checks:")
for section in sections:
    print(f"  {'✓' if section in readme else '✗'}  {section}")

print(f"\n  Total characters: {len(readme)}")
print("\nrequirements.txt and README.md check passed.")

=== requirements.txt ===
  Exists: True
pandas>=2.0.0
numpy>=1.24.0
vaderSentiment>=3.3.2
lingua-language-detector>=2.0.0
deep-translator>=1.11.0
thefuzz>=0.20.0
python-Levenshtein>=0.21.0
openpyxl>=3.1.0
faiss-cpu>=1.7.4
sentence-transformers>=2.2.0
Package checks:
  ✓  pandas
  ✓  numpy
  ✓  vaderSentiment
  ✓  lingua-language-detector
  ✓  deep-translator
  ✓  thefuzz
  ✓  python-Levenshtein
  ✓  openpyxl
  ✓  faiss-cpu
  ✓  sentence-transformers

=== README.md ===
  Exists: True

Section checks:
  ✓  Quick Start
  ✓  File Structure
  ✓  Final Column Schema
  ✗  Dashboard Master CSV
  ✗  Semantic Search
  ✓  Language Detection and Translation
  ✓  Emoji Handling
  ✓  Fuzzy deduplication
  ✓  FUZZY_MATCH_THRESHOLD
  ✓  YYYYMMDD
  ✗  Run Order
  ✓  How To Add A New Data Source
  ✓  How To Update
  ✓  Dependencies
  ✓  GOOGLE_SCRAPE_DATE
  ✗  faiss-cpu
  ✗  sentence-transformers
  ✗  semantic_best_score
  ✗  semantic_theme

  Total characters: 11373

requirements.txt and README.md chec